# pydantic to structure gemini output

In [2]:
from dotenv import load_dotenv
import os
from google import genai


# load_dotenv()

client = genai.Client() # api_key=os.getenv("GEMINI_API_KEY"))

# response = client.models.generate_content(
#     model="gemini-2.5-flash", contents="tell a gbg joke"
# )
# # response.text
# print(response.text)

In [3]:
def ask_llm(prompt):

    response = client.models.generate_content(
        model="gemini-2.5-flash", 
        contents=prompt
    )

    return response.text

In [3]:
ask_llm("tell me a secret")

'As an AI, I don\'t have personal secrets. Secrets are usually about private experiences, thoughts, or actions that someone chooses to keep hidden.\n\nSince I don\'t have a personal life, feelings, or experiences in the way humans do, I don\'t have anything to keep secret! My "knowledge" comes from the vast amount of text data I\'ve been trained on, and my purpose is to share information, not hide it.\n\nBut if you\'d like, I can tell you a surprising fact, a riddle, or a fun piece of trivia!'

# try to get data

In [4]:
response = ask_llm("""
    Du är en expert inom köp och sälj av bostäder, likt en proffsig mäklare.
    Generera bostadspriser, månadsavgifter, address, stad, boarea i jsonformat (ej markdown)

    Exempel:
            {
                "address": "Fågelvägen 5,
                "price_sek": 3000000,
                "city": "Göteborg",
                "monthly_fee": 4000,
                "area": 60
            }   
                   
    Ge mig en lista på 5 bostäder
""")

response

'[\n    {\n        "address": "Karlavägen 75",\n        "price_sek": 7850000,\n        "city": "Stockholm",\n        "monthly_fee": 4100,\n        "area": 75\n    },\n    {\n        "address": "Linnégatan 23B",\n        "price_sek": 4500000,\n        "city": "Göteborg",\n        "monthly_fee": 3950,\n        "area": 68\n    },\n    {\n        "address": "Fersens Väg 9",\n        "price_sek": 3200000,\n        "city": "Malmö",\n        "monthly_fee": 3700,\n        "area": 80\n    },\n    {\n        "address": "Klostergatan 12A",\n        "price_sek": 2980000,\n        "city": "Uppsala",\n        "monthly_fee": 3400,\n        "area": 55\n    },\n    {\n        "address": "Västra Esplanaden 15",\n        "price_sek": 1950000,\n        "city": "Umeå",\n        "monthly_fee": 2900,\n        "area": 48\n    }\n]'

In [5]:
print(response)

[
    {
        "address": "Karlavägen 75",
        "price_sek": 7850000,
        "city": "Stockholm",
        "monthly_fee": 4100,
        "area": 75
    },
    {
        "address": "Linnégatan 23B",
        "price_sek": 4500000,
        "city": "Göteborg",
        "monthly_fee": 3950,
        "area": 68
    },
    {
        "address": "Fersens Väg 9",
        "price_sek": 3200000,
        "city": "Malmö",
        "monthly_fee": 3700,
        "area": 80
    },
    {
        "address": "Klostergatan 12A",
        "price_sek": 2980000,
        "city": "Uppsala",
        "monthly_fee": 3400,
        "area": 55
    },
    {
        "address": "Västra Esplanaden 15",
        "price_sek": 1950000,
        "city": "Umeå",
        "monthly_fee": 2900,
        "area": 48
    }
]


## parse and validate data

In [6]:
from pydantic import BaseModel, Field
import json 

class Apartment(BaseModel):
    address: str 
    city: str 
    price_sek: int = Field(gt=1000000, lt = 8000000) 
    monthly_fee: int 
    area: int 

class ApartmentList(BaseModel):
    objects: list[Apartment]


apartments = ApartmentList.model_validate({"objects": json.loads(response)})
apartments

ApartmentList(objects=[Apartment(address='Karlavägen 75', city='Stockholm', price_sek=7850000, monthly_fee=4100, area=75), Apartment(address='Linnégatan 23B', city='Göteborg', price_sek=4500000, monthly_fee=3950, area=68), Apartment(address='Fersens Väg 9', city='Malmö', price_sek=3200000, monthly_fee=3700, area=80), Apartment(address='Klostergatan 12A', city='Uppsala', price_sek=2980000, monthly_fee=3400, area=55), Apartment(address='Västra Esplanaden 15', city='Umeå', price_sek=1950000, monthly_fee=2900, area=48)])

In [7]:
apartments.objects

[Apartment(address='Karlavägen 75', city='Stockholm', price_sek=7850000, monthly_fee=4100, area=75),
 Apartment(address='Linnégatan 23B', city='Göteborg', price_sek=4500000, monthly_fee=3950, area=68),
 Apartment(address='Fersens Väg 9', city='Malmö', price_sek=3200000, monthly_fee=3700, area=80),
 Apartment(address='Klostergatan 12A', city='Uppsala', price_sek=2980000, monthly_fee=3400, area=55),
 Apartment(address='Västra Esplanaden 15', city='Umeå', price_sek=1950000, monthly_fee=2900, area=48)]

In [8]:
apartments.objects[1].address, apartments.objects[1].city

('Linnégatan 23B', 'Göteborg')

In [9]:
addresses = [apartment.address for apartment in apartments.objects ]

addresses

['Karlavägen 75',
 'Linnégatan 23B',
 'Fersens Väg 9',
 'Klostergatan 12A',
 'Västra Esplanaden 15']

In [10]:
addresses = [
    apartment.address
    for apartment in apartments.objects
    if apartment.price_sek < 4000000
]

addresses

['Fersens Väg 9', 'Klostergatan 12A', 'Västra Esplanaden 15']

## filter out adress, city, price, monthly_fee for the interval 4m-8m

In [11]:
import pandas as pd



apartments_price_range = [
    [apartment.address, apartment.city, apartment.price_sek, apartment.monthly_fee]
    for apartment
    in apartments.objects
    if 4000000 < apartment.price_sek < 8000000
    # if home.price_sek > 4000000 and home.price_sek < 8000000
]

apartments_price_range



# price_limited_apartments

[['Karlavägen 75', 'Stockholm', 7850000, 4100],
 ['Linnégatan 23B', 'Göteborg', 4500000, 3950]]

## convert to df

In [12]:
# Filter
filtered_apartments = [
    apartment for apartment in apartments.objects if 4_000_000 < apartment.price_sek < 8_000_000
]

filtered_apartments

[Apartment(address='Karlavägen 75', city='Stockholm', price_sek=7850000, monthly_fee=4100, area=75),
 Apartment(address='Linnégatan 23B', city='Göteborg', price_sek=4500000, monthly_fee=3950, area=68)]

In [13]:
df_filtered = pd.DataFrame(
    [
        apartments.model_dump(include={"address", "city", "price_sek", "monthly_fee"})
        for apartments in filtered_apartments
    ]
)
df_filtered

,address,city,price_sek,monthly_fee
0,Karlavägen 75,Stockholm,7850000,4100
1,Linnégatan 23B,Göteborg,4500000,3950


In [14]:
df_filtered.to_csv("filtered_apartments.csv", index=False)

pandas dataframe alternative way

In [28]:
apartments.objects

[Apartment(address='Karlavägen 75', city='Stockholm', price_sek=7850000, monthly_fee=4100, area=75),
 Apartment(address='Linnégatan 23B', city='Göteborg', price_sek=4500000, monthly_fee=3950, area=68),
 Apartment(address='Fersens Väg 9', city='Malmö', price_sek=3200000, monthly_fee=3700, area=80),
 Apartment(address='Klostergatan 12A', city='Uppsala', price_sek=2980000, monthly_fee=3400, area=55),
 Apartment(address='Västra Esplanaden 15', city='Umeå', price_sek=1950000, monthly_fee=2900, area=48)]

In [35]:
addresses = [apartment.address for apartment in apartments.objects]
areas = [apartment.area for apartment in apartments.objects]
prices = [apartment.price_sek for apartment in apartments.objects]
cities = [apartment.city for apartment in apartments.objects]
monthly_fees = [apartment.monthly_fee for apartment in apartments.objects]

df = pd.DataFrame({"address": addresses,"area": areas, "price": prices, "monthly_fee": monthly_fees, "city": cities })
df.to_csv("apartments.csv", index=False)

### import to duckbd alternative

In [47]:
import duckdb
print(duckdb.__version__)
# db_table = duckdb.read_csv("filtered_apartments.csv")
# print(db_table)


1.3.2


In [18]:
# 1. Anslut till din DuckDB-databas (skapar filen om den inte finns)
# con = duckdb.connect("apartments.duckdb")
# bash command: duckdb -ui "apartments.duckdb"

## import to duckdb with dlt

In [46]:
import dlt

@dlt.resource(write_disposition="replace", table_name="apartment")

def load_data():
    yield df

pipeline = dlt.pipeline(
    pipeline_name="apartments",
    destination="duckdb",
    dataset_name="staging"
)
load_info = pipeline.run(load_data())
print(load_info)


Pipeline apartments load step completed in 0.14 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:///c:\Users\metal\Skola\course\ai_engineering_stefan_lundberg\code-alongs\07_b_pydentic_gemini\apartments.duckdb location to store data
Load package 1757334314.2926276 is LOADED and contains no failed jobs
